[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus_optimization/03_gradient_descent_mechanics/first_principles.ipynb)

# Topic 03: Gradient Descent Mechanics

## 1. First-Principles Intuition & Motivation

Every trained neural network in existence is the output of one line of arithmetic executed a few million times:

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \eta \nabla f(\mathbf{x}_k).
$$

Read literally, the rule says: *measure the direction of steepest increase, walk the opposite way, and control your ambition with a single scalar $\eta$*. Nothing in that sentence guarantees anything. Whether the sequence converges, how fast, and to what, are all questions the update rule itself cannot answer — they are answered by the geometry of $f$ near the trajectory, and by whether $\eta$ respects that geometry.

This notebook derives those answers from scratch. The strategy is the one Topic 02 set up: replace the unknowable $f$ by a *local model with a certified error bound*, then reason about the model. The descent lemma supplies the certificate, the quadratic case supplies exact solutions, and the two together explain the entire observed phenomenology of training curves — smooth descent, oscillation, plateaus, divergence.

### Three regimes of one scalar

Run gradient descent on the one-dimensional quadratic $f(x) = \frac{1}{2}\lambda x^2$ with $\lambda \gt 0$. The update is exactly solvable: $x_{k+1} = (1 - \eta\lambda)x_k$, so $x_k = (1-\eta\lambda)^k x_0$. Everything hinges on the single number $r = 1 - \eta\lambda$:

| Range of $\eta$ | $r = 1-\eta\lambda$ | Behavior of $x_k$ |
|---|---|---|
| $0 \lt \eta \lt 1/\lambda$ | $r \in (0,1)$ | monotone decay to $0$ |
| $\eta = 1/\lambda$ | $r = 0$ | exact convergence in one step |
| $1/\lambda \lt \eta \lt 2/\lambda$ | $r \in (-1,0)$ | sign-alternating decay (visible oscillation, still stable) |
| $\eta = 2/\lambda$ | $r = -1$ | perpetual oscillation, no progress |
| $\eta \gt 2/\lambda$ | $r \lt -1$ | geometric divergence |

A training curve that oscillates but still trends down is not broken — it is running in the third row. A training curve that explodes has crossed $2/\lambda$. In $d$ dimensions the same table applies *simultaneously to every eigendirection of the Hessian*, and the tightest constraint wins: $\eta \lt 2/\lambda_{\max}$.

### The continuous shadow: gradient flow

Divide the update by $\eta$ and let $\eta \to 0$:

$$
\frac{\mathbf{x}_{k+1} - \mathbf{x}_k}{\eta} = -\nabla f(\mathbf{x}_k) \quad \longrightarrow \quad \dot{\mathbf{x}}(t) = -\nabla f(\mathbf{x}(t)).
$$

Gradient descent is *explicit Euler integration of the gradient-flow ODE with time step $\eta$*. The continuous flow can never increase $f$, because $\frac{d}{dt}f(\mathbf{x}(t)) = -\lVert \nabla f(\mathbf{x}(t)) \rVert_2^2 \le 0$ — no assumptions, no step-size condition. Therefore **every learning-rate pathology is a discretization artifact**, not a property of the descent idea. This single reframing explains why implicit/proximal methods have no stability limit, why "warmup" resembles a small initial time step for a stiff ODE, and why the ratio $\eta\lambda_{\max}$ (a dimensionless *Courant number*) is the quantity that decides stability.

### Notation used throughout

- $f: \mathbb{R}^d \to \mathbb{R}$ — the objective; $f^\star = \inf f$, and $\mathbf{x}^\star$ a minimizer when one exists.
- $\mathbf{x}_k$ — the iterate at step $k$; $\mathbf{e}_k = \mathbf{x}_k - \mathbf{x}^\star$ the error; $\delta_k = f(\mathbf{x}_k) - f^\star$ the optimality gap.
- $\eta \gt 0$ — the step size (learning rate); $L$ — the smoothness constant; $\mu$ — the strong-convexity constant.
- $H = \nabla^2 f$ — the Hessian; $\lambda_{\min} \le \cdots \le \lambda_{\max}$ its eigenvalues; $\kappa = \lambda_{\max}/\lambda_{\min}$ (equivalently $L/\mu$) the condition number.
- $\rho(A)$ — spectral radius of $A$; $\lVert \cdot \rVert_2$ the Euclidean norm, $\lVert \cdot \rVert_{\mathrm{op}}$ the operator norm.
- $g_k$ — a stochastic gradient estimate at step $k$ with $\mathbb{E}[g_k] = \nabla f(\mathbf{x}_k)$ and variance bounded by $\sigma^2$.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 2.1 (Gradient descent).** Given $f$ differentiable, $\mathbf{x}_0 \in \mathbb{R}^d$, and a step-size sequence $\eta_k \gt 0$, gradient descent generates

$$
\mathbf{x}_{k+1} = \mathbf{x}_k - \eta_k \nabla f(\mathbf{x}_k), \qquad k = 0, 1, 2, \dots
$$

With $\eta_k \equiv \eta$ the method is *constant-step*; with $\eta_k$ chosen to (approximately) minimize $\phi(\eta) = f(\mathbf{x}_k - \eta\nabla f(\mathbf{x}_k))$ it is *steepest descent with (exact) line search*.

**Definition 2.2 ($L$-smoothness).** $f$ is $L$-smooth if $\lVert \nabla f(\mathbf{x}) - \nabla f(\mathbf{y}) \rVert_2 \le L\lVert \mathbf{x} - \mathbf{y} \rVert_2$ for all $\mathbf{x}, \mathbf{y}$; for $f \in C^2$ this is equivalent to $\lambda_{\max}(\nabla^2 f) \le L$ everywhere. Topic 02 proved the consequence used relentlessly below, the **descent lemma**:

$$
f(\mathbf{y}) \le f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) + \frac{L}{2}\lVert \mathbf{y}-\mathbf{x} \rVert_2^2 .
$$

**Definition 2.3 (Convexity and $\mu$-strong convexity).** $f$ is convex if for all $\mathbf{x},\mathbf{y}$, $f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x})$; it is $\mu$-strongly convex ($\mu \gt 0$) if

$$
f(\mathbf{y}) \ge f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) + \frac{\mu}{2}\lVert \mathbf{y}-\mathbf{x} \rVert_2^2 .
$$

For $f \in C^2$: convexity means $\nabla^2 f \succeq 0$, strong convexity means $\nabla^2 f \succeq \mu I$. Together with $L$-smoothness the spectrum is boxed: $\mu I \preceq \nabla^2 f \preceq L I$, and $\kappa = L/\mu \ge 1$.

**Theorem 2.4 (Sufficient decrease).** If $f$ is $L$-smooth and $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\nabla f(\mathbf{x}_k)$, then

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \eta\left(1 - \frac{L\eta}{2}\right)\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 .
$$

The coefficient is positive exactly when $0 \lt \eta \lt 2/L$ and is maximized at $\eta = 1/L$, where it equals $\frac{1}{2L}$.

**Theorem 2.5 (Exact dynamics and stability on quadratics).** For $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top H\mathbf{x} - \mathbf{b}^\top \mathbf{x}$ with $H = H^\top \succ 0$ and $\mathbf{x}^\star = H^{-1}\mathbf{b}$, constant-step gradient descent obeys

$$
\mathbf{e}_{k+1} = (I - \eta H)\,\mathbf{e}_k, \qquad \mathbf{e}_k = (I-\eta H)^k \mathbf{e}_0 ,
$$

which converges from every $\mathbf{x}_0$ if and only if $\rho(I - \eta H) \lt 1$, i.e. if and only if $0 \lt \eta \lt 2/\lambda_{\max}$.

**Theorem 2.6 (Optimal constant step and the $\kappa$-rate).** For the same quadratic, the step minimizing the worst-case contraction factor is

$$
\eta^\star = \frac{2}{\lambda_{\min} + \lambda_{\max}}, \qquad \rho^\star = \frac{\lambda_{\max}-\lambda_{\min}}{\lambda_{\max}+\lambda_{\min}} = \frac{\kappa - 1}{\kappa + 1},
$$

so $\lVert \mathbf{e}_k \rVert_2 \le (\rho^\star)^k \lVert \mathbf{e}_0 \rVert_2$ and reaching accuracy $\epsilon$ costs $O\!\left(\kappa \log(1/\epsilon)\right)$ iterations.

**Theorem 2.7 (Convergence rates for constant step $\eta = 1/L$).**

1. *(Nonconvex, $L$-smooth, $f$ bounded below.)* $\displaystyle \min_{0 \le t \lt k} \lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le \frac{2L\left(f(\mathbf{x}_0)-f^\star\right)}{k}$, so the smallest gradient norm decays like $O(1/\sqrt{k})$.
2. *(Convex, $L$-smooth.)* $\displaystyle f(\mathbf{x}_k) - f^\star \le \frac{L\lVert \mathbf{x}_0 - \mathbf{x}^\star \rVert_2^2}{2k}$ — sublinear $O(1/k)$.
3. *($\mu$-strongly convex, $L$-smooth.)* $\displaystyle f(\mathbf{x}_k) - f^\star \le \left(1 - \frac{1}{\kappa}\right)^k \left(f(\mathbf{x}_0)-f^\star\right)$ — linear (geometric) convergence with $\kappa = L/\mu$.

**Definition 2.8 (Momentum methods).** Heavy ball (Polyak) and Nesterov accelerated gradient are, respectively,

$$
\mathbf{v}_{k+1} = \beta\mathbf{v}_k - \eta\nabla f(\mathbf{x}_k), \quad \mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{v}_{k+1};
$$

$$
\mathbf{y}_k = \mathbf{x}_k + \beta(\mathbf{x}_k - \mathbf{x}_{k-1}), \quad \mathbf{x}_{k+1} = \mathbf{y}_k - \eta\nabla f(\mathbf{y}_k).
$$

On strongly convex quadratics, tuned heavy ball attains the contraction $\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$, i.e. $O(\sqrt{\kappa}\log(1/\epsilon))$ iterations; Nesterov attains $O(\sqrt{\kappa}\log(1/\epsilon))$ on all smooth strongly convex $f$ and $O(1/k^2)$ on smooth convex $f$, matching Nesterov's lower bound for first-order methods.

**Definition 2.9 (Minibatch stochastic gradient descent).** For a finite-sum objective $f(\mathbf{x}) = \frac{1}{n}\sum_{i=1}^{n} f_i(\mathbf{x})$, draw a minibatch $B$ of size $B$ uniformly and use

$$
g_k = \frac{1}{B}\sum_{i \in B} \nabla f_i(\mathbf{x}_k), \qquad \mathbb{E}[g_k \mid \mathbf{x}_k] = \nabla f(\mathbf{x}_k), \qquad \mathbb{E}\left[\lVert g_k - \nabla f(\mathbf{x}_k) \rVert_2^2\right] \le \frac{\sigma^2}{B},
$$

where $\sigma^2$ bounds the per-example gradient variance (sampling with replacement).

**Theorem 2.10 (SGD noise floor, constant step).** If $f$ is $\mu$-strongly convex and $L$-smooth, $g_k$ unbiased with variance $\le \sigma_B^2 = \sigma^2/B$, and $0 \lt \eta \le 1/L$, then

$$
\mathbb{E}\left[f(\mathbf{x}_k) - f^\star\right] \le (1-\eta\mu)^k\left(f(\mathbf{x}_0)-f^\star\right) + \frac{\eta L \sigma_B^2}{2\mu}.
$$

Constant-step SGD converges *linearly to a noise ball of radius $\propto \eta\sigma^2/B$*, not to the minimizer. Robbins–Monro schedules $\sum_k \eta_k = \infty$, $\sum_k \eta_k^2 \lt \infty$ shrink the ball to zero.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: Exact error dynamics and the stability threshold $\eta \lt 2/\lambda_{\max}$

*Claim.* (Theorem 2.5) For $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top H\mathbf{x} - \mathbf{b}^\top\mathbf{x}$, $H \succ 0$, gradient descent satisfies $\mathbf{e}_k = (I-\eta H)^k\mathbf{e}_0$, and $\mathbf{x}_k \to \mathbf{x}^\star$ for every $\mathbf{x}_0$ iff $0 \lt \eta \lt 2/\lambda_{\max}$.

*Proof.* **Step 1 (linearize the error).** The gradient is $\nabla f(\mathbf{x}) = H\mathbf{x} - \mathbf{b}$. Since $H\mathbf{x}^\star = \mathbf{b}$, we may rewrite it purely in terms of the error: $\nabla f(\mathbf{x}_k) = H(\mathbf{x}_k - \mathbf{x}^\star) = H\mathbf{e}_k$. Subtract $\mathbf{x}^\star$ from both sides of the update:

$$
\mathbf{e}_{k+1} = \mathbf{x}_{k+1} - \mathbf{x}^\star = \mathbf{x}_k - \eta H \mathbf{e}_k - \mathbf{x}^\star = (I - \eta H)\mathbf{e}_k .
$$

Iterating gives $\mathbf{e}_k = (I-\eta H)^k \mathbf{e}_0$: gradient descent on a quadratic is a *linear* dynamical system, with no approximation whatsoever.

**Step 2 (diagonalize).** $H$ is symmetric, so $H = Q\Lambda Q^\top$ with $Q$ orthogonal and $\Lambda = \operatorname{diag}(\lambda_1,\dots,\lambda_d)$. Put $\tilde{\mathbf{e}}_k = Q^\top \mathbf{e}_k$. Since $Q^\top(I-\eta H)Q = I - \eta\Lambda$ is diagonal, the coupled system splits into $d$ independent scalar recursions:

$$
\tilde{e}_{k,i} = (1 - \eta\lambda_i)^k\, \tilde{e}_{0,i}, \qquad i = 1,\dots,d .
$$

**Step 3 (stability).** $\lVert \mathbf{e}_k \rVert_2^2 = \sum_i (1-\eta\lambda_i)^{2k}\tilde{e}_{0,i}^2$ (orthogonal $Q$ preserves norms). This tends to $0$ for *every* $\mathbf{e}_0$ iff $\lvert 1 - \eta\lambda_i \rvert \lt 1$ for every $i$, i.e. $0 \lt \eta\lambda_i \lt 2$ for every $i$. With all $\lambda_i \gt 0$ the binding constraint is the largest:

$$
0 \lt \eta \lt \frac{2}{\lambda_{\max}} . \qquad \blacksquare
$$

*Reading the result.* Mode $i$ contracts by $\lvert 1-\eta\lambda_i \rvert$ per step. High-curvature modes are the ones that blow up; low-curvature modes are the ones that crawl ($1 - \eta\lambda_{\min} \approx 1$). A single $\eta$ must serve both — this tension *is* ill-conditioning, and no choice of scalar $\eta$ removes it.

### Proof 3.2: The optimal constant step and the $(\kappa-1)/(\kappa+1)$ rate

*Claim.* (Theorem 2.6) The minimizer of $\rho(\eta) = \max_i \lvert 1-\eta\lambda_i \rvert$ is $\eta^\star = \frac{2}{\lambda_{\min}+\lambda_{\max}}$, with optimal value $\frac{\kappa-1}{\kappa+1}$.

*Proof.* **Step 1 (only the extreme eigenvalues matter).** For fixed $\eta \gt 0$, the map $\lambda \mapsto \lvert 1-\eta\lambda \rvert$ is convex in $\lambda$, so its maximum over the interval $[\lambda_{\min},\lambda_{\max}]$ — which contains all $\lambda_i$ — is attained at an endpoint:

$$
\rho(\eta) = \max\left\{ \lvert 1-\eta\lambda_{\min} \rvert,\ \lvert 1-\eta\lambda_{\max} \rvert \right\}.
$$

**Step 2 (minimize a max of two lines).** On the useful range $0 \lt \eta \lt 2/\lambda_{\max}$ we have $1-\eta\lambda_{\min} \gt 0$, so the first term is $1-\eta\lambda_{\min}$: *decreasing* in $\eta$. The second term is $\lvert 1-\eta\lambda_{\max} \rvert$, which decreases to $0$ at $\eta = 1/\lambda_{\max}$ and then *increases* as $\eta\lambda_{\max}-1$. The max of a decreasing and a V-shaped function is minimized where the decreasing branch meets the rising branch:

$$
1 - \eta\lambda_{\min} = \eta\lambda_{\max} - 1 \quad \Longrightarrow \quad \eta^\star = \frac{2}{\lambda_{\min}+\lambda_{\max}} .
$$

**Step 3 (the value).** Substituting,

$$
\rho^\star = 1 - \frac{2\lambda_{\min}}{\lambda_{\min}+\lambda_{\max}} = \frac{\lambda_{\max}-\lambda_{\min}}{\lambda_{\max}+\lambda_{\min}} = \frac{\kappa-1}{\kappa+1}, \qquad \kappa = \frac{\lambda_{\max}}{\lambda_{\min}} .
$$

**Step 4 (iteration complexity).** To reach $\lVert \mathbf{e}_k \rVert_2 \le \epsilon\lVert \mathbf{e}_0 \rVert_2$ we need $(\rho^\star)^k \le \epsilon$, i.e. $k \ge \dfrac{\log(1/\epsilon)}{\log(1/\rho^\star)}$. For large $\kappa$, $\rho^\star = 1 - \frac{2}{\kappa+1}$ and $\log(1/\rho^\star) \approx \frac{2}{\kappa}$, hence

$$
k = O\!\left(\frac{\kappa}{2}\log\frac{1}{\epsilon}\right) = O\!\left(\kappa\log\frac{1}{\epsilon}\right). \qquad \blacksquare
$$

*The zig-zag, made precise.* At $\eta^\star$ the two extreme modes contract by the *same* factor but with *opposite signs*: $1-\eta^\star\lambda_{\min} = +\rho^\star$ and $1-\eta^\star\lambda_{\max} = -\rho^\star$. The steep direction flips sign every step while the flat direction creeps forward — exactly the bouncing-across-the-valley picture, and it is optimal behavior, not a bug. Doubling $\kappa$ doubles the iteration count.

### Proof 3.3: Sufficient decrease, and why $\eta = 1/L$ is the canonical choice

*Claim.* (Theorem 2.4) If $f$ is $L$-smooth, then $f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \eta\left(1-\frac{L\eta}{2}\right)\lVert \nabla f(\mathbf{x}_k) \rVert_2^2$.

*Proof.* Apply the descent lemma (Topic 02, Theorem 2.5) with $\mathbf{x} = \mathbf{x}_k$ and $\mathbf{y} = \mathbf{x}_{k+1} = \mathbf{x}_k - \eta\nabla f(\mathbf{x}_k)$, so that $\mathbf{y}-\mathbf{x} = -\eta\nabla f(\mathbf{x}_k)$:

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) + \nabla f(\mathbf{x}_k)^\top\left(-\eta\nabla f(\mathbf{x}_k)\right) + \frac{L}{2}\left\lVert \eta\nabla f(\mathbf{x}_k) \right\rVert_2^2 .
$$

The middle term is $-\eta\lVert \nabla f(\mathbf{x}_k) \rVert_2^2$ and the last is $\frac{L\eta^2}{2}\lVert \nabla f(\mathbf{x}_k) \rVert_2^2$. Collecting,

$$
f(\mathbf{x}_{k+1}) \le f(\mathbf{x}_k) - \eta\left(1 - \frac{L\eta}{2}\right)\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 . \qquad \blacksquare
$$

*Three corollaries, all immediate.*

**(a) The stability window from smoothness alone.** The bracket $c(\eta) = \eta\left(1-\frac{L\eta}{2}\right)$ is positive exactly for $0 \lt \eta \lt 2/L$ — the general-$f$ version of the quadratic threshold $\eta \lt 2/\lambda_{\max}$, since $L = \lambda_{\max}$ for a quadratic.

**(b) The best guaranteed step.** $c'(\eta) = 1 - L\eta = 0$ at $\eta = 1/L$, where $c(1/L) = \frac{1}{2L}$. So

$$
f\!\left(\mathbf{x}_k - \tfrac{1}{L}\nabla f(\mathbf{x}_k)\right) \le f(\mathbf{x}_k) - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_k) \rVert_2^2 .
$$

Equivalently: $\eta = 1/L$ is the argmin of the *certified upper model* $f(\mathbf{x}_k) + \nabla f^\top \mathbf{h} + \frac{L}{2}\lVert \mathbf{h}\rVert_2^2$ — a principled step, not a tuned one.

**(c) Nonconvex convergence to stationarity.** Sum (b) over $t = 0,\dots,k-1$ and telescope, using $f(\mathbf{x}_k) \ge f^\star$:

$$
\frac{1}{2L}\sum_{t=0}^{k-1}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le f(\mathbf{x}_0) - f(\mathbf{x}_k) \le f(\mathbf{x}_0) - f^\star .
$$

Since the minimum of $k$ terms is at most their average,

$$
\min_{0 \le t \lt k}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2 \le \frac{2L\left(f(\mathbf{x}_0)-f^\star\right)}{k},
$$

i.e. $\min_t \lVert \nabla f(\mathbf{x}_t) \rVert_2 = O(1/\sqrt{k})$. No convexity was used: for *any* bounded-below smooth loss, including a deep network's, gradient descent drives the gradient to zero at a certified rate. It says nothing about *which* stationary point (Topic 04).

### Proof 3.4: $O(1/k)$ convergence for convex $L$-smooth $f$

*Claim.* (Theorem 2.7.2) With $\eta = 1/L$ and $f$ convex and $L$-smooth with minimizer $\mathbf{x}^\star$,

$$
f(\mathbf{x}_k) - f^\star \le \frac{L\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2}{2k} .
$$

*Proof.* Write $\delta_t = f(\mathbf{x}_t)-f^\star$ and $\mathbf{g}_t = \nabla f(\mathbf{x}_t)$.

**Step 1 (distance recursion).** Expand the squared distance after one step:

$$
\lVert \mathbf{x}_{t+1}-\mathbf{x}^\star \rVert_2^2 = \lVert \mathbf{x}_t - \mathbf{x}^\star \rVert_2^2 - 2\eta\, \mathbf{g}_t^\top(\mathbf{x}_t-\mathbf{x}^\star) + \eta^2\lVert \mathbf{g}_t \rVert_2^2 .
$$

**Step 2 (convexity bounds the inner product).** The first-order convexity inequality at $\mathbf{x}_t$ evaluated at $\mathbf{x}^\star$ gives $f^\star \ge f(\mathbf{x}_t) + \mathbf{g}_t^\top(\mathbf{x}^\star - \mathbf{x}_t)$, i.e.

$$
\mathbf{g}_t^\top(\mathbf{x}_t - \mathbf{x}^\star) \ge f(\mathbf{x}_t) - f^\star = \delta_t .
$$

**Step 3 (smoothness bounds the gradient norm).** By Proof 3.3(b) with $\eta = 1/L$: $\delta_{t+1} \le \delta_t - \frac{1}{2L}\lVert \mathbf{g}_t \rVert_2^2$, hence $\lVert \mathbf{g}_t \rVert_2^2 \le 2L(\delta_t - \delta_{t+1})$, and with $\eta^2 = 1/L^2$,

$$
\eta^2\lVert \mathbf{g}_t \rVert_2^2 \le \frac{2}{L}\left(\delta_t - \delta_{t+1}\right).
$$

**Step 4 (combine).** Substituting Steps 2–3 into Step 1 with $\eta = 1/L$:

$$
\lVert \mathbf{x}_{t+1}-\mathbf{x}^\star \rVert_2^2 \le \lVert \mathbf{x}_t-\mathbf{x}^\star \rVert_2^2 - \frac{2}{L}\delta_t + \frac{2}{L}\left(\delta_t-\delta_{t+1}\right) = \lVert \mathbf{x}_t-\mathbf{x}^\star \rVert_2^2 - \frac{2}{L}\delta_{t+1} .
$$

**Step 5 (telescope).** Sum over $t = 0,\dots,k-1$; the distances telescope and the left-most term is dropped as nonnegative:

$$
\frac{2}{L}\sum_{t=1}^{k}\delta_t \le \lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2 .
$$

**Step 6 (monotonicity).** Sufficient decrease makes $\delta_1 \ge \delta_2 \ge \cdots \ge \delta_k$, so $k\,\delta_k \le \sum_{t=1}^k \delta_t \le \frac{L}{2}\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2$, giving

$$
\delta_k \le \frac{L\lVert \mathbf{x}_0-\mathbf{x}^\star \rVert_2^2}{2k} . \qquad \blacksquare
$$

*What $O(1/k)$ feels like.* Halving the error requires *doubling the total iteration count*: getting from $10^{-2}$ to $10^{-3}$ costs ten times all the work done so far. Nesterov acceleration improves the constant to $O(1/k^2)$, which is optimal for first-order methods on this class — no gradient-only algorithm can do better in the worst case.

### Proof 3.5: Linear convergence under strong convexity

*Claim.* (Theorem 2.7.3) If $f$ is $\mu$-strongly convex and $L$-smooth and $\eta = 1/L$, then $\delta_k \le \left(1-\frac{1}{\kappa}\right)^k \delta_0$ with $\kappa = L/\mu$.

*Proof.* **Step 1 (strong convexity implies the Polyak–Łojasiewicz inequality).** Minimize both sides of the strong-convexity lower bound over $\mathbf{y}$. The right side, $q(\mathbf{y}) = f(\mathbf{x}) + \nabla f(\mathbf{x})^\top(\mathbf{y}-\mathbf{x}) + \frac{\mu}{2}\lVert \mathbf{y}-\mathbf{x} \rVert_2^2$, is a strictly convex quadratic minimized at $\mathbf{y} = \mathbf{x} - \frac{1}{\mu}\nabla f(\mathbf{x})$ with value $f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x}) \rVert_2^2$. Since $f(\mathbf{y}) \ge q(\mathbf{y})$ for all $\mathbf{y}$, taking the min of the left side gives

$$
f^\star \ge f(\mathbf{x}) - \frac{1}{2\mu}\lVert \nabla f(\mathbf{x}) \rVert_2^2 \quad \Longleftrightarrow \quad \lVert \nabla f(\mathbf{x}) \rVert_2^2 \ge 2\mu\left(f(\mathbf{x}) - f^\star\right).
$$

This is the PL inequality: *a small gradient certifies near-optimality*, the property that fails badly on nonconvex losses with plateaus.

**Step 2 (plug into sufficient decrease).** From Proof 3.3(b), $\delta_{t+1} \le \delta_t - \frac{1}{2L}\lVert \nabla f(\mathbf{x}_t) \rVert_2^2$. Applying Step 1,

$$
\delta_{t+1} \le \delta_t - \frac{2\mu}{2L}\delta_t = \left(1 - \frac{\mu}{L}\right)\delta_t = \left(1-\frac{1}{\kappa}\right)\delta_t .
$$

**Step 3 (iterate).** By induction $\delta_k \le \left(1-\frac{1}{\kappa}\right)^k\delta_0$. $\blacksquare$

**Iteration complexity.** Using $1-x \le e^{-x}$, $\delta_k \le e^{-k/\kappa}\delta_0$, so $\delta_k \le \epsilon$ once

$$
k \ge \kappa \log\frac{\delta_0}{\epsilon} = O\!\left(\kappa\log\frac{1}{\epsilon}\right),
$$

the same $\kappa$-scaling that Proof 3.2 derived exactly on quadratics — reassuring, since a strongly convex smooth function is sandwiched between two quadratics with curvatures $\mu$ and $L$. Note the qualitative jump: $O(1/k)$ needed $1/\epsilon$ iterations, this needs only $\log(1/\epsilon)$. Strong convexity — in practice supplied by $\ell_2$ regularization, which adds $\lambda I$ to the Hessian — is what turns a crawl into a geometric contraction, and simultaneously *lowers* $\kappa$ from $L/\mu$ to $(L+\lambda)/(\mu+\lambda)$.

### Proof 3.6: Gradient flow always descends; discretization is what breaks

*Claim.* (a) Along $\dot{\mathbf{x}} = -\nabla f(\mathbf{x})$, $f$ is non-increasing and strictly decreasing away from stationary points. (b) On the quadratic $f = \frac{1}{2}\mathbf{x}^\top H\mathbf{x}$ the exact flow is $\mathbf{x}(t) = e^{-tH}\mathbf{x}_0$, which decays for *every* $t \gt 0$, while explicit Euler replaces the mode factor $e^{-\eta\lambda}$ by $1-\eta\lambda$ and thereby manufactures the stability limit.

*Proof.* **(a)** By the chain rule,

$$
\frac{d}{dt}f(\mathbf{x}(t)) = \nabla f(\mathbf{x}(t))^\top \dot{\mathbf{x}}(t) = -\lVert \nabla f(\mathbf{x}(t)) \rVert_2^2 \le 0,
$$

with equality iff $\nabla f(\mathbf{x}(t)) = 0$. So $f$ is a Lyapunov function for the flow: no step size, no smoothness constant, no convexity is needed. $\blacksquare$

**(b)** For $f = \frac{1}{2}\mathbf{x}^\top H\mathbf{x}$ the ODE is linear, $\dot{\mathbf{x}} = -H\mathbf{x}$, with solution $\mathbf{x}(t) = e^{-tH}\mathbf{x}_0$. In the eigenbasis, mode $i$ evolves as $e^{-\lambda_i t}$, which for $\lambda_i \gt 0$ decays monotonically for all $t$, however large. Explicit Euler with step $\eta$ produces instead the factor $1-\eta\lambda_i$ — the first two terms of $e^{-\eta\lambda_i} = 1 - \eta\lambda_i + \frac{(\eta\lambda_i)^2}{2} - \cdots$. The truncation is harmless while $\eta\lambda_i \lesssim 1$ but catastrophic once $\eta\lambda_i \gt 2$, where $\lvert 1-\eta\lambda_i \rvert \gt 1 \gt e^{-\eta\lambda_i}$. $\blacksquare$

*Consequences worth carrying.*

- **Stiffness = ill-conditioning.** A stiff ODE is one whose eigenvalues span many orders of magnitude; the explicit step is capped by the fastest mode while the solution time is set by the slowest. That is verbatim the $\kappa$ story, imported from numerical analysis (Topic 03 of `numerical_methods`).
- **Implicit Euler has no limit.** The backward step $\mathbf{x}_{k+1} = \mathbf{x}_k - \eta\nabla f(\mathbf{x}_{k+1})$ gives mode factor $(1+\eta\lambda)^{-1}$, which is in $(0,1)$ for *every* $\eta \gt 0$: unconditionally stable. This is exactly the proximal-point method, $\mathbf{x}_{k+1} = \arg\min_{\mathbf{z}}\left\{ f(\mathbf{z}) + \frac{1}{2\eta}\lVert \mathbf{z}-\mathbf{x}_k \rVert_2^2 \right\}$, and it is why proximal and implicit methods tolerate huge steps at the cost of solving a subproblem.
- **Momentum is a second-order ODE.** Heavy ball is a discretization of $\ddot{\mathbf{x}} + a\dot{\mathbf{x}} + \nabla f(\mathbf{x}) = 0$ — a *damped particle* rather than an overdamped one, which is why it can coast through narrow valleys instead of bouncing.

## 4. Computational & Algorithmic Insights

### 4.1 Diagnosing a training run from the theory

The three theorems above give a decision procedure that costs almost nothing to run:

| Observation | Diagnosis | Theory |
|---|---|---|
| Loss increases, then NaN | $\eta \gt 2/\lambda_{\max}$; some mode has factor of magnitude greater than $1$ | Proof 3.1 |
| Loss oscillates with an overall downward trend | $1 \lt \eta\lambda_{\max} \lt 2$: top modes alternate but contract | Proof 3.1 |
| Loss drops fast then flattens on a long plateau | Fast modes converged; progress now paced by $1-\eta\lambda_{\min}$ | Proof 3.2 |
| Halving $\eta$ roughly doubles time-to-target | You are in the stable regime; the bottleneck is $\kappa$, not $\eta$ | Proof 3.2 |
| Gradient norm falls like $k^{-1/2}$ but loss keeps drifting | Nonconvex stationarity rate, no optimality guarantee | Proof 3.3(c) |

A cheap probe of $\lambda_{\max}$: run a few power iterations with Hessian-vector products $\nabla^2 f(\mathbf{x})\mathbf{v} = \nabla\!\left(\nabla f(\mathbf{x})^\top\mathbf{v}\right)$, which cost one extra backward pass (Topic 01) and never form the Hessian.

### 4.2 Momentum: buying $\sqrt{\kappa}$ with one extra buffer

On the quadratic, heavy ball turns the scalar recursion $\tilde{e}_{k+1} = (1-\eta\lambda)\tilde{e}_k$ into the two-term recursion

$$
\tilde{e}_{k+1} = (1 + \beta - \eta\lambda)\tilde{e}_k - \beta\,\tilde{e}_{k-1},
$$

whose behavior is governed by the roots of $z^2 - (1+\beta-\eta\lambda)z + \beta = 0$. When the discriminant is negative, both roots are complex with modulus exactly $\sqrt{\beta}$ — the contraction factor becomes *independent of $\lambda$*. Choosing $\beta$ so that this holds for every $\lambda \in [\mu, L]$ gives

$$
\beta^\star = \left(\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}\right)^2, \qquad \eta^\star = \frac{4}{\left(\sqrt{L}+\sqrt{\mu}\right)^2}, \qquad \text{rate } \sqrt{\beta^\star} = \frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}.
$$

For $\kappa = 10^4$: plain gradient descent needs $\sim \kappa = 10^4$ iterations per digit-decade, momentum needs $\sim\sqrt{\kappa} = 10^2$ — a hundredfold saving from one extra velocity buffer. Nesterov's variant evaluates the gradient at the *look-ahead* point $\mathbf{y}_k$, which supplies a correction term that extends the guarantee from quadratics to all smooth convex functions and yields the optimal $O(1/k^2)$ rate.

Practical notes: $\beta \in [0.9, 0.99]$ corresponds to an effective step multiplier $\frac{1}{1-\beta} \in [10, 100]$, so raising $\beta$ without lowering $\eta$ is a hidden learning-rate increase; and momentum's benefit is largest precisely where curvature is most anisotropic.

### 4.3 Minibatch noise, batch size, and schedules

**Where the noise comes from.** Sampling $B$ examples i.i.d. makes $g_k$ an unbiased estimate whose covariance scales as $\Sigma/B$. Two consequences follow from Theorem 2.10:

- **The noise ball.** With constant $\eta$, SGD converges linearly only until $\delta_k \approx \frac{\eta L\sigma^2}{2\mu B}$, then random-walks inside that ball. Loss curves flattening at a nonzero level is usually this, not a bad minimum.
- **Two ways to shrink the ball.** Halve $\eta$, or double $B$: the bound depends on the ratio $\eta/B$. This is the theory behind *"decaying the learning rate is equivalent to increasing the batch size"* (Smith et al., 2018) and behind the **linear scaling rule** $\eta \propto B$ used to keep large-batch training equivalent to small-batch training (Goyal et al., 2017) — valid while $\eta$ stays below the stability ceiling $2/L$, which is why linear scaling breaks at very large $B$ and needs warmup.

**Schedules, and what each one is for.**

| Schedule | Form | Rationale |
|---|---|---|
| Robbins–Monro | $\sum_k \eta_k = \infty$, $\sum_k \eta_k^2 \lt \infty$ (e.g. $\eta_k \propto 1/k$) | Provably shrinks the noise ball to zero; asymptotically optimal, practically too slow early |
| Step decay | $\eta \to \eta/10$ at milestones | Each drop shrinks the noise ball by $10\times$; the classic staircase loss curve |
| Cosine | $\eta_k = \frac{\eta_0}{2}\left(1+\cos\frac{\pi k}{K}\right)$ | Smooth anneal to zero, no milestone tuning |
| Warmup | $\eta$ ramps up over the first few thousand steps | Early curvature estimates and $L$ are unreliable; keeps steps inside the trust radius of the linear model (Topic 02) |
| Cyclical / restarts | periodic re-raising of $\eta$ | Deliberately re-injects energy to escape narrow basins (Topic 04) |

**Adaptive methods** (Adagrad, RMSProp, Adam) replace the scalar $\eta$ by a per-coordinate step $\eta/\sqrt{\hat{v}_i}$, an inexpensive diagonal preconditioner: they attack $\kappa$ directly rather than tolerating it. The `../../optimization/` module develops them in full.

### 4.4 Where the gradient actually comes from: backpropagation

Gradient descent consumes $\nabla f(\mathbf{x}_k)$ as if it were free. It is not — but it is remarkably cheap. For a feedforward network $\mathbf{a}^{(l)} = \phi\!\left(W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}\right)$ with scalar loss $\mathcal{L}$, define the adjoint $\boldsymbol{\delta}^{(l)} = \partial\mathcal{L}/\partial \mathbf{z}^{(l)}$ where $\mathbf{z}^{(l)}$ is the pre-activation. The chain rule gives the backward recursion

$$
\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{z}^{(L)}}\mathcal{L}, \qquad \boldsymbol{\delta}^{(l)} = \left(W^{(l+1)}\right)^\top \boldsymbol{\delta}^{(l+1)} \odot \phi'\!\left(\mathbf{z}^{(l)}\right),
$$

$$
\frac{\partial \mathcal{L}}{\partial W^{(l)}} = \boldsymbol{\delta}^{(l)}\left(\mathbf{a}^{(l-1)}\right)^\top, \qquad \frac{\partial\mathcal{L}}{\partial \mathbf{b}^{(l)}} = \boldsymbol{\delta}^{(l)} .
$$

Three points that matter for the mechanics of descent:

- **Cost.** Reverse-mode automatic differentiation computes the full gradient of a scalar loss with respect to *all* $d$ parameters at a cost of $O(1)$ forward passes (about $2\times$–$3\times$), independent of $d$. Forward-mode would cost $O(d)$. This asymmetry is the reason first-order methods dominate at billions of parameters.
- **Memory.** The backward pass needs the stored activations $\mathbf{a}^{(l-1)}$, so memory scales with depth $\times$ batch — the constraint that gradient checkpointing trades back for compute.
- **Conditioning.** The product of Jacobians $\prod_l \left(W^{(l)}\right)^\top \operatorname{diag}\phi'$ is what makes gradients vanish or explode; it is also what makes the loss Hessian's spectrum span many decades, i.e. it is the *source* of the $\kappa$ that Proofs 3.2 and 3.5 price. Normalization layers, residual connections, and careful initialization are, in this language, conditioning devices.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Physics: overdamped motion, relaxation, and stiff integration

Gradient flow $\dot{\mathbf{x}} = -\nabla U(\mathbf{x})$ is the equation of a particle in a potential $U$ in the **overdamped limit** — inertia negligible, velocity proportional to force, as for a bead in honey. Near a stable equilibrium $U$ is quadratic (Topic 02) and the motion is a sum of exponential relaxations $e^{-\lambda_i t}$ with time constants $\tau_i = 1/\lambda_i$; a system with widely separated $\tau_i$ is *stiff*, and explicit integrators must take steps of order the smallest $\tau$ while the physics evolves on the largest — a spread measured by exactly $\kappa$.

Adding inertia gives $m\ddot{\mathbf{x}} + c\dot{\mathbf{x}} = -\nabla U(\mathbf{x})$, whose discretization is heavy-ball momentum; the optimal $\beta$ corresponds to *critical damping*, the setting a mechanical engineer picks to reach equilibrium fastest without ringing. Adding thermal noise gives Langevin dynamics $d\mathbf{x} = -\nabla U\,dt + \sqrt{2T}\,d\mathbf{W}$, the physics counterpart of SGD: the "temperature" is set by $\eta\sigma^2/B$, and the stationary distribution $\propto e^{-U/T}$ makes precise the folklore that larger $\eta$ or smaller $B$ makes the optimizer prefer wide basins (Topic 04).

### 5.2 ML: the training loop is these theorems

Consider a concrete supervised setup: ridge-regularized least squares $f(\mathbf{w}) = \frac{1}{n}\lVert X\mathbf{w}-\mathbf{y} \rVert_2^2 + \lambda\lVert \mathbf{w} \rVert_2^2$. Its Hessian is the constant matrix $\frac{2}{n}X^\top X + 2\lambda I$, so every quantity in this notebook is computable in closed form: $L = \frac{2}{n}\sigma_{\max}(X)^2 + 2\lambda$, $\mu = \frac{2}{n}\sigma_{\min}(X)^2 + 2\lambda$, safe steps are $\eta \lt 2/L$, the best step is $2/(\mu+L)$, and convergence is geometric with rate $(\kappa-1)/(\kappa+1)$. Feature standardization shrinks $\kappa$ and is therefore an *optimization* intervention as much as a statistical one; ridge $\lambda$ buys strong convexity and caps $\kappa$ at $(L_0+2\lambda)/2\lambda$.

For deep networks nothing is computable in closed form, yet the same five knobs organize practice: the **stability ceiling** (why the loss diverges above some $\eta$, and the "edge of stability" phenomenon where training hovers at $\eta\lambda_{\max}\approx 2$), the **condition number** (why normalization, residual connections, and Adam help), **momentum** (why $\beta \approx 0.9$ is a default), **noise** (why the loss plateaus at a floor set by $\eta/B$, and why decaying $\eta$ produces the characteristic staircase), and **schedules** (warmup, cosine, restarts). The legacy notebook [`../gradient_descent.ipynb`](../gradient_descent.ipynb) runs each of these experiments end to end.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source | Where |
|---|---|---|
| Update rule, ill-conditioning, deep-net optimization practice | Goodfellow, Bengio & Courville, *Deep Learning* | Ch. 4.3, Ch. 8 |
| Descent lemma, $O(1/k)$, linear rate, optimal $O(1/k^2)$ and lower bounds | Nesterov, *Lectures on Convex Optimization* | Ch. 2.1–2.2 |
| Steepest descent, exact/backtracking line search, condition-number geometry | Boyd & Vandenberghe, *Convex Optimization* | Ch. 9.1–9.3 |
| Line-search theory, Wolfe conditions, convergence of steepest descent | Nocedal & Wright, *Numerical Optimization* | Ch. 3 |
| Heavy ball and the $\sqrt{\kappa}$ rate | Polyak (1964), *Some Methods of Speeding up the Convergence of Iteration Methods* | §2 |
| Eigendecoupled visualization of GD and momentum | Goh, *Why Momentum Really Works* (Distill, 2017) | whole |
| SGD analysis, noise ball, batch-size effects | Bottou, Curtis & Nocedal, *Optimization Methods for Large-Scale ML* (SIAM Review, 2018) | §4–5 |
| Linear scaling rule and warmup at scale | Goyal et al. (2017), *Accurate, Large Minibatch SGD* | §2–5 |
| Decay $\eta$ or grow $B$: the equivalence | Smith, Kindermans & Le (2018), *Don't Decay the Learning Rate, Increase the Batch Size* | §2 |
| Reverse-mode AD supplying the gradient | Baydin et al. (2018), *Automatic Differentiation in ML: a Survey* (JMLR) | §3 |

**Backward pointers**: the descent lemma and the local-model viewpoint come from [`../02_taylor_approximation_and_local_models/`](../02_taylor_approximation_and_local_models/); the gradients themselves come from [`../01_derivatives_and_gradients_for_ml/`](../01_derivatives_and_gradients_for_ml/).

**Forward pointers**: Topic 04 asks *where* these iterations end up — convexity, saddle points, and the shape of deep-learning landscapes. Momentum, Adam, and stochastic methods are developed in depth in [`../../optimization/`](../../optimization/); runnable experiments for every claim here are in [`../gradient_descent.ipynb`](../gradient_descent.ipynb).